# MAMoE-50 Kaggle Training Pipeline

Notebook ini mengeksekusi **2 Fase Pipeline** utama dari arsitektur *Memory-Augmented Mixture-of-Experts* (MAMoE-50):
- **Fase 1 (Pre-training)**: Pembelajaran konseptual (Next Token Predictor) menggunakan dataset CSV *VQFat/Cosmopedia*.
- **Fase 2 (Fine-tuning)**: Penyelarasan instruksi percakapan menggunakan dataset Parquet *T5Gemma2 Chat*.

In [ ]:
# 1. Setup Environment (Instalasi Dependensi JAX)
!pip install --upgrade jax jaxlib flax optax pandas pyarrow fastparquet -q
!nvidia-smi

In [ ]:
# 2. Clone Repositori MAMoE-50
!git clone https://github.com/Akhyar11/memorybank.git
%cd memorybank

--- 
## FASE 1: PRE-TRAINING (Next Token Predictor)
Fase ini bertujuan agar model memahami pola dasar bahasa Indonesia. Model akan dilatih tanpa mempedulikan struktur percakapan terlebih dahulu.

In [ ]:
# Pastikan dataset VQFat CSV tersedia dari /kaggle/input
import os
if os.path.exists('/kaggle/input/vqfat-indonesian-corpus/vqfat_cosmopedia_id.csv'):
    print("✅ Dataset VQFat Ditemukan untuk Fase 1!")
else:
    print("⚠️ DATASET VQFAT TIDAK DITEMUKAN. Tambahkan dari Kaggle Datasets.")

# Update batch size ke skala Multi-GPU untuk T4 x2
!sed -i 's/batch_size = 2/batch_size = 16/g' train.py
!sed -i 's/seq_len = 64/seq_len = 1024/g' train.py

In [ ]:
# Eksekusi Pre-Training (Multi-GPU JAX pmap)
!python train.py

--- 
## FASE 2: FINE-TUNING (Chat Alignment)
Fase ini bertujuan agar model dapat bertindak sebagai asisten AI yang responsif dengan mengambil *checkpoints* (bobot model) dari Fase 1, lalu melatihnya ulang menggunakan dataset format obrolan (*Chat-formatted Parquet*).

In [ ]:
# Pastikan dataset T5Gemma2 Parquet tersedia dari /kaggle/input
if os.path.exists('/kaggle/input/t5gemma2-indonesia-chat/t5gemma2_chat_multiturn.parquet'):
    print("✅ Dataset Parquet T5Gemma Ditemukan untuk Fase 2!")
else:
    print("⚠️ DATASET T5GEMMA TIDAK DITEMUKAN. Tambahkan dari Kaggle Datasets.")

In [ ]:
# Eksekusi Fine-Tuning
# Skrip ini secara internal akan me-load weights dari Fase 1 dan menggunakan Learning Rate yang lebih kecil (5e-5)
!python finetune.py